In [12]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [13]:
!kaggle datasets download -d salader/dogsvscats

Dataset URL: https://www.kaggle.com/datasets/salader/dogsvscats
License(s): unknown
dogsvscats.zip: Skipping, found more recently modified local copy (use --force to force download)


In [14]:
import zipfile
zip_ref=zipfile.ZipFile('/content/dogsvscats.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [16]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Conv2D,Dense,MaxPooling2D,Flatten,BatchNormalization,Dropout,Input
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import cv2

# **Loading Data**

In [17]:
train_ds=keras.utils.image_dataset_from_directory(
    directory='/content/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)
validation_ds=keras.utils.image_dataset_from_directory(
    directory='/content/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


# **Scaling**

In [18]:
train_ds=train_ds.map(lambda x,y:(x/255.0,y))

In [19]:
validation_ds=validation_ds.map(lambda x,y:(x/255.0,y))

# **CNN Model**

In [32]:
model=Sequential([
    Input(shape=(256,256,3)),


    Conv2D(32,kernel_size=(3,3),strides=(1,1),padding='valid',activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),


    Conv2D(64,kernel_size=(3,3),strides=(1,1),padding='valid',activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),


    Conv2D(128,kernel_size=(3,3),strides=(1,1),padding='valid',activation='relu'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),


    Flatten(),

    Dense(128,activation='relu'),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

In [33]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['Accuracy'])

In [34]:
callback=EarlyStopping(
    monitor='val_loss',
    mode='min',
    verbose=1,
    min_delta=0.01,
    patience=20,
    restore_best_weights=True
)

In [35]:
history=model.fit(train_ds,epochs=100,verbose=1,callbacks=callback,validation_data=validation_ds)

Epoch 1/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 67s 96ms/step - Accuracy: 0.5448 - loss: 1.5900 - val_Accuracy: 0.5284 - val_loss: 0.7480
Epoch 2/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 92ms/step - Accuracy: 0.6035 - loss: 0.6716 - val_Accuracy: 0.5868 - val_loss: 0.6672
Epoch 3/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 93ms/step - Accuracy: 0.6682 - loss: 0.6164 - val_Accuracy: 0.6694 - val_loss: 0.6277
Epoch 4/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 94ms/step - Accuracy: 0.7286 - loss: 0.5407 - val_Accuracy: 0.7214 - val_loss: 0.5556
Epoch 5/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 98ms/step - Accuracy: 0.7685 - loss: 0.4825 - val_Accuracy: 0.7124 - val_loss: 0.5439
Epoch 6/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - Accuracy: 0.7948 - loss: 0.4361 - val_Accuracy: 0.7740 - val_loss: 0.4881
Epoch 7/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - Accuracy: 0.8288 - loss: 0.3794 - val_Accuracy: 0.8096 - val_loss: 0.4084
Epoch 8/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 98ms/step - Accuracy: 0.8516 - loss: 0